In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Helper per l'apprendimento del rumore

<details>
<summary><b>Versioni dei pacchetti</b></summary>

Il codice in questa pagina è stato sviluppato usando i seguenti requisiti.
Si consiglia di utilizzare queste versioni o versioni più recenti.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>

Le tecniche di mitigazione degli errori [PEA](./error-mitigation-and-suppression-techniques#probabilistic-error-amplification-pea) e [PEC](./error-mitigation-and-suppression-techniques#probabilistic-error-cancellation-pec) utilizzano entrambe una componente di apprendimento del rumore basata su un [modello di rumore di Pauli-Lindblad](https://arxiv.org/abs/2201.09866), che viene tipicamente gestita durante l'esecuzione dopo aver inviato uno o più job tramite `qiskit-ibm-runtime`, senza alcun accesso locale al modello di rumore addestrato. Tuttavia, a partire da `qiskit-ibm-runtime` 0.27.1, sono state create le classi [`NoiseLearner`](../api/qiskit-ibm-runtime/noise-learner) e [`NoiseLearnerOptions`](../api/qiskit-ibm-runtime/options-noise-learner-options) per ottenere i risultati di questi esperimenti di apprendimento del rumore. I risultati possono poi essere salvati localmente come `NoiseLearnerResult` e usati come input in esperimenti successivi. Questa pagina fornisce una panoramica del suo utilizzo e delle opzioni disponibili.

## Panoramica
La classe `NoiseLearner` esegue esperimenti che caratterizzano i processi di rumore basandosi su un modello di rumore di Pauli-Lindblad per uno o più circuiti. Dispone di un metodo `run()` che esegue gli esperimenti di apprendimento e accetta in input una lista di circuiti oppure un [PUB](./primitive-input-output#overview-of-pubs), restituendo un `NoiseLearnerResult` contenente i canali di rumore appresi e i metadati relativi ai job inviati. Di seguito è riportato uno snippet di codice che illustra l'utilizzo del programma helper.

In [1]:
from qiskit import QuantumCircuit
from qiskit.transpiler import CouplingMap
from qiskit.transpiler import generate_preset_pass_manager

from qiskit_ibm_runtime import QiskitRuntimeService, EstimatorV2
from qiskit_ibm_runtime.noise_learner import NoiseLearner
from qiskit_ibm_runtime.options import (
    NoiseLearnerOptions,
    ResilienceOptionsV2,
    EstimatorOptions,
)

# Build a circuit with two entangling layers
num_qubits = 27
edges = list(CouplingMap.from_line(num_qubits, bidirectional=False))
even_edges = edges[::2]
odd_edges = edges[1::2]

circuit = QuantumCircuit(num_qubits)
for pair in even_edges:
    circuit.cx(pair[0], pair[1])
for pair in odd_edges:
    circuit.cx(pair[0], pair[1])

# Choose a backend to run on
service = QiskitRuntimeService()
backend = service.least_busy()

# Transpile the circuit for execution
pm = generate_preset_pass_manager(backend=backend, optimization_level=3)
circuit_to_learn = pm.run(circuit)

# Instantiate a NoiseLearner object and execute the noise learning program
learner = NoiseLearner(mode=backend)
job = learner.run([circuit_to_learn])
noise_model = job.result()

Il risultante `NoiseLearnerResult.data` è una lista di oggetti [`LayerError`](../api/qiskit-ibm-runtime/utils-noise-learner-result-layer-error) contenente il [modello di rumore](https://arxiv.org/abs/2201.09866) per ogni singolo layer di entanglement appartenente ai circuiti target. Ogni `LayerError` memorizza le informazioni del layer, sotto forma di circuito e insieme di etichette di qubit, insieme al [`PauliLindbladError`](../api/qiskit-ibm-runtime/utils-noise-learner-result-pauli-lindblad-error) per il modello di rumore appreso per il layer in questione.

In [2]:
print(
    f"Noise learner result contains {len(noise_model.data)} entries"
    f" and has the following type:\n {type(noise_model)}\n"
)
print(
    f"Each element of `NoiseLearnerResult` then contains"
    f" an object of type:\n {type(noise_model.data[0])}\n"
)
print(
    f"And each of these `LayerError` objects possess"
    f" data on the generators for the error channel: \n{noise_model.data[0].error.generators}\n"
)
print(f"Along with the error rates: \n{noise_model.data[0].error.rates}\n")

Noise learner result contains 2 entries and has the following type:
 <class 'qiskit_ibm_runtime.utils.noise_learner_result.NoiseLearnerResult'>

Each element of `NoiseLearnerResult` then contains an object of type:
 <class 'qiskit_ibm_runtime.utils.noise_learner_result.LayerError'>

And each of these `LayerError` objects possess data on the generators for the error channel: 
['IIIIIIIIIIIIIIIIIIIIIIIIIIX', 'IIIIIIIIIIIIIIIIIIIIIIIIIIY',
 'IIIIIIIIIIIIIIIIIIIIIIIIIIZ', 'IIIIIIIIIIIIIIIIIIIIIIIIIXI',
 'IIIIIIIIIIIIIIIIIIIIIIIIIXX', 'IIIIIIIIIIIIIIIIIIIIIIIIIXY',
 'IIIIIIIIIIIIIIIIIIIIIIIIIXZ', 'IIIIIIIIIIIIIIIIIIIIIIIIIYI',
 'IIIIIIIIIIIIIIIIIIIIIIIIIYX', 'IIIIIIIIIIIIIIIIIIIIIIIIIYY',
 'IIIIIIIIIIIIIIIIIIIIIIIIIYZ', 'IIIIIIIIIIIIIIIIIIIIIIIIIZI',
 'IIIIIIIIIIIIIIIIIIIIIIIIIZX', 'IIIIIIIIIIIIIIIIIIIIIIIIIZY',
 'IIIIIIIIIIIIIIIIIIIIIIIIIZZ', 'IIIIIIIIIIIIIIIIIIIIIIIIXII',
 'IIIIIIIIIIIIIIIIIIIIIIIIXXI', 'IIIIIIIIIIIIIIIIIIIIIIIIXYI',
 'IIIIIIIIIIIIIIIIIIIIIIIIXZI', 'IIIIIIIIIIIIIIIIIIIIII

The `LayerError.error` attribute of the noise learning result contains the generators and error rates of the fitted Pauli Lindblad model, which has the form

$$ \Lambda(\rho) = \exp{\sum_j r_j \left(P_j \rho P_j^\dagger - \rho\right)}, $$

where the $r_j$ are the `LayerError.rates` and $P_j$ are the Pauli operators specified in `LayerError.generators`.

## Noise learning options

You can choose among several options to input when you instantiate a `NoiseLearner` object. These options are encapsulated by the `qiskit_ibm_runtime.options.NoiseLearnerOptions` class and include the ability to specify the maximum layers to learn, number of randomizations, and the twirling strategy, among others. Refer to the API documentation about [`NoiseLearnerOptions`](../api/qiskit-ibm-runtime/options-noise-learner-options) for more detailed information.

Below is a simple example showing how to use the `NoiseLearnerOptions` in a `NoiseLearner` experiment:

In [3]:
# Build a GHZ circuit
circuit = QuantumCircuit(10)
circuit.h(0)
circuit.cx(range(0, 9), range(1, 10))
# Choose a backend to run on
service = QiskitRuntimeService()
backend = service.least_busy()

# Transpile the circuit for execution
pm = generate_preset_pass_manager(backend=backend, optimization_level=3)
circuit_to_run = pm.run(circuit_to_learn)

# Instantiate a noise learner options object
learner_options = NoiseLearnerOptions(
    max_layers_to_learn=3, num_randomizations=32, twirling_strategy="all"
)

# Instantiate a NoiseLearner object and execute the noise learning program
learner = NoiseLearner(mode=backend, options=learner_options)
job = learner.run([circuit_to_run])
noise_model = job.result()

L'attributo `LayerError.error` del risultato dell'apprendimento del rumore contiene i generatori e i tassi di errore del modello di Pauli-Lindblad adattato, che ha la forma

$$ \Lambda(\rho) = \exp{\sum_j r_j \left(P_j \rho P_j^\dagger - \rho\right)}, $$

dove gli $r_j$ sono i `LayerError.rates` e i $P_j$ sono gli operatori di Pauli specificati in `LayerError.generators`.

## Opzioni per l'apprendimento del rumore
Puoi scegliere tra diverse opzioni da specificare quando istanzi un oggetto `NoiseLearner`. Queste opzioni sono racchiuse nella classe `qiskit_ibm_runtime.options.NoiseLearnerOptions` e includono la possibilità di specificare il numero massimo di layer da apprendere, il numero di randomizzazioni e la strategia di twirling, tra le altre. Per informazioni più dettagliate, consulta la documentazione API su [`NoiseLearnerOptions`](../api/qiskit-ibm-runtime/options-noise-learner-options).

Di seguito è riportato un semplice esempio che mostra come usare `NoiseLearnerOptions` in un esperimento con `NoiseLearner`:

In [4]:
# pass the noise model to the `estimator.options` attribute directly
estimator = EstimatorV2(mode=backend)
estimator.options.resilience.layer_noise_model = noise_model

In [5]:
# Specify options via a ResilienceOptionsV2 object
resilience_options = ResilienceOptionsV2(layer_noise_model=noise_model)
estimator_options = EstimatorOptions(resilience=resilience_options)
estimator = EstimatorV2(mode=backend, options=estimator_options)

In [6]:
# Specify options via a dictionary
options_dict = {
    "resilience_level": 2,
    "resilience": {"layer_noise_model": noise_model},
}

estimator = EstimatorV2(mode=backend, options=options_dict)

Once the noise model is passed into the `EstimatorV2` object, it can be used to run workloads and perform error mitigation as normal.

## Next steps

<Admonition type="tip" title="Recommendations">
    - Read more about [configuring error mitigation](configure-error-mitigation).
    - Review the [EstimatorOptions API reference](/docs/api/qiskit-ibm-runtime/options-estimator-options) and [ResilienceOptionsV2 API reference](/docs/api/qiskit-ibm-runtime/options-resilience-options-v2).
    - Learn more about [Error mitigation and suppression techniques](error-mitigation-and-suppression-techniques) that are available through Qiskit Runtime.
    - Review how to [Specify options](specify-runtime-options) for the Qiskit Runtime primitives.
    - Read [Migrate to V2 primitives](/docs/guides/v2-primitives).
</Admonition>